<a href="https://colab.research.google.com/github/jmcelroy01/agentic-ai-for-functional-materials-discovery/blob/main/01_anatomy_of_a_molecular_crystal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1 · Anatomy of a molecular crystal  (~8 min)

Paracetamol has two well-known crystal forms:

| | Form I | Form II |
|---|---|---|
| Space group | P2₁/n (monoclinic) | Pbca (orthorhombic) |
| Stability at room T | stable (the tablet form) | metastable |
| Tabletting | poor (brittle) | good (has slip planes) |

Structures from the Crystallography Open Database (COD 7126913 and 7105574, 100 K).

In [1]:
# @title 0 · Setup  (run this cell first; ~1-2 min on Colab)
import os, sys
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/csp-tutorial"):
        !git clone -q {REPO} /content/csp-tutorial
    else:
        !git -C /content/csp-tutorial pull -q
    %cd /content/csp-tutorial
    !pip -q install pyxtal mace-torch py3Dmol pymatgen rdkit "requests==2.32.4"
import importlib, csp_utils as cu
importlib.reload(cu)   # picks up a fresh copy if Setup is run again
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ready")

/content/csp-tutorial
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ready


## 1.1 Look at the two crystals
Drag to rotate, scroll to zoom. The brown box is one unit cell.

In [2]:
form1 = cu.load_cif(cu.DATA / "paracetamol_form_I.cif")
form2 = cu.load_cif(cu.DATA / "paracetamol_form_II.cif")
print(form1.get_chemical_formula(), "|", form2.get_chemical_formula())
cu.show(form1, supercell=(2, 2, 1))

C32H36N4O8 | C64H72N8O16


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
cu.show(form2, supercell=(1, 2, 1))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 1.2 The numbers that describe a packing

* Z: molecules per unit cell; Z′: molecules in the asymmetric unit (1 for both forms)
* density and packing coefficient (fraction of space filled; organic crystals are usually 0.65–0.77)
* hydrogen bonds: two donors (O–H, N–H) and two acceptors (O=C, O–H)

In [4]:
table = pd.DataFrame([cu.describe(form1, name="Form I"), cu.describe(form2, name="Form II")]).set_index("name").T
table

name,Form I,Form II
space group,P2_1/c (#14),Pbca (#61)
"a,b,c (A)",7.09 / 9.21 / 11.60,11.77 / 7.19 / 17.17
"alpha,beta,gamma",90.0 / 97.9 / 90.0,90.0 / 90.0 / 90.0
Z (molecules/cell),4,8
volume/molecule (A^3),187.5,181.5
density (g/cm3),1.339,1.383
packing coefficient,0.754,0.779
H-bonds/cell,8,16


In [5]:
pd.DataFrame(cu.hydrogen_bonds(form1)).query("intermolecular")

,donor,acceptor,H_A,angle,intermolecular
0,O0,O11,1.85,166.3,True
1,O1,O10,1.85,166.3,True
2,O2,O9,1.85,166.3,True
3,O3,O8,1.85,166.3,True
4,N12,O3,2.09,157.1,True
5,N13,O2,2.09,157.1,True
6,N14,O1,2.09,157.1,True
7,N15,O0,2.09,157.1,True


> **Question.** Which form is denser? Form I is the stable one. Does paracetamol follow the density rule?

## 1.3 Which form is lower in energy?

Lattice energy per molecule:

$$E_\text{latt} = \frac{E_\text{crystal}}{Z} - E_\text{gas molecule}$$

Energies from MACE-OFF, a machine-learned force field for organic molecules (Kovács *et al.*, *JACS* 2025).
Both structures were relaxed with it beforehand.

In [6]:
from ase.io import read
relaxed = {n: read(cu.DATA / f"paracetamol_form_{n}_relaxed.extxyz") for n in ("I", "II")}
e_gas = cu.gas_phase_energy()
rows = []
for n, at in relaxed.items():
    e = cu.energy_per_molecule(at)
    rows.append({"form": n, "E_latt (kJ/mol)": round(e - e_gas, 1), "density relaxed": round(cu.density(at), 3),
                 "density exp (100 K)": round(cu.density(form1 if n == "I" else form2), 3)})
pd.DataFrame(rows)

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


,form,E_latt (kJ/mol),density relaxed,density exp (100 K)
0,I,-125.4,1.348,1.339
1,II,-122.8,1.384,1.383


## 1.4 Squeeze the crystal

Why does each form have the density it has? It sits at the bottom of an energy well: push the molecules closer
and they clash, pull them apart and they lose their contacts. Make the cell smaller or bigger, keep every molecule
rigid, and watch the energy. Each new slider position is one MACE-OFF calculation.

In [7]:
from ipywidgets import interact, FloatSlider

curve = {"I": {}, "II": {}}
big = cu.squeeze(relaxed["I"], 1.25)
_, frame = plt.subplots(); LIM = cu.draw_cell(big, frame); plt.close()

@interact(scale=FloatSlider(value=1.0, min=0.95, max=1.25, step=0.01, continuous_update=False, description="cell scale"))
def squeeze(scale):
    scale = round(scale, 2)
    for n, at in relaxed.items():
        if scale not in curve[n]:
            s = cu.squeeze(at, scale)
            curve[n][scale] = (cu.density(s), cu.energy_per_molecule(s) - e_gas)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
    for n, c in (("I", "#893F0D"), ("II", "#C9A27E")):
        ax.plot(*zip(*sorted(curve[n].values())), "o-", c=c, label=f"Form {n}")
        ax.plot(*curve[n][scale], "o", ms=14, mfc="none", mec="k")
    low = min(e for d in curve.values() for _, e in d.values())
    ax.set_ylim(low - 5, low + 150)
    ax.set_xlabel("density (g/cm$^3$)"); ax.set_ylabel("E_latt (kJ/mol)"); ax.legend()
    cu.draw_cell(cu.squeeze(relaxed["I"], scale), ax2, title=f"Form I, cell × {scale:.2f}", limits=LIM)
    plt.show()

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='cell scale', max=1.25, min=…

> **Question.** Why does the energy go up so much faster when you squeeze than when you stretch?